In [ ]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

warnings.filterwarnings("ignore")

from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
df = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
df = df[df.los>=1]
df.shape

(74829, 8)

In [3]:
d_labitems = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_labitems.csv.gz')
d_labitems = d_labitems.fillna('None')

In [11]:
labs = [51279,51493, 52151,50802,52038,51463,51300,51516,52407,51249, 53185, 51277, 52172, 53165, 53166, 50811]
labs = pd.unique(labs)
len(labs)

16

In [13]:
dataset_path = 'D:/mimic-iv-3.1/mimiciv/3.1/hosp/labevents.csv.gz'

chunksize = 10000000
lab=pd.DataFrame()
for labevents in tqdm(pd.read_csv(dataset_path, compression='gzip', header=0, index_col=None,chunksize=chunksize)):
    labevents=labevents[['subject_id', 'itemid', 'hadm_id', 'charttime', 'storetime', 'value', 'valueuom', 'flag']]
    labevents=labevents[labevents['hadm_id'].isin(df['hadm_id'].unique())]
    labevents=labevents[labevents['itemid'].isin(labs)]
    labevents=labevents.dropna(subset=['value'])

    lab=lab.append(labevents, ignore_index=True)

16it [07:43, 28.97s/it]


In [15]:
lab = pd.merge(lab,df[['hadm_id','stay_id', 'intime', 'outtime']],on='hadm_id',how='left')

In [16]:
lab['intime'] = pd.to_datetime(lab['intime'])
lab['outtime'] = pd.to_datetime(lab['outtime'])
lab['charttime'] = pd.to_datetime(lab['charttime'])
lab['storetime'] = pd.to_datetime(lab['storetime'])

In [17]:
within_criteria = lab['charttime'].between(lab['intime'],lab['outtime'])
within = lab[within_criteria]

In [18]:
within['event_time_from_admit'] = within['charttime'] - within['intime']
within['start_time']=within['event_time_from_admit'].dt.total_seconds() / 3600
within['los'] = (within['outtime'] - within['intime']).dt.total_seconds() / 3600
within=within[within['start_time']>=0]
within.head(2)

,subject_id,itemid,hadm_id,charttime,storetime,value,valueuom,flag,stay_id,intime,outtime,event_time_from_admit,start_time,los
0,10000690,51249,25860671.0,2150-11-03 02:56:00,2150-11-03 03:46:00,33.2,%,NaN,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,0 days 07:19:00,7.316667,93.438056
1,10000690,51277,25860671.0,2150-11-03 02:56:00,2150-11-03 03:46:00,16.4,%,abnormal,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,0 days 07:19:00,7.316667,93.438056


In [19]:
within_72 = within[within['start_time']<=72]
within_72.shape

(1434957, 14)

In [20]:
num_col = [51300,50802,50811,51249,51277,51279,51493,51516,52172]
num_col

[51300, 50802, 50811, 51249, 51277, 51279, 51493, 51516, 52172]

In [21]:
str_col = [51463]
str_col

[51463]

In [22]:
len(list(set(num_col)|set(str_col)))

10

In [ ]:
def compute_outlier_imputation(arr, upper_percentile, lower_percentile):

    arr = pd.to_numeric(arr, errors='coerce')

    upper_bound = np.percentile(arr, upper_percentile)
    lower_bound = np.percentile(arr, lower_percentile)
    

    arr[arr < lower_bound] = lower_bound
    arr[arr > upper_bound] = upper_bound

    return arr

def outlier_imputation(data, id_attribute, value_attribute, num_col, upper_percentile=98, lower_percentile=2):

    grouped = data.groupby(id_attribute)


    for id_number, group in grouped:

        if id_number not in num_col:
            continue
        
        index = group.index
        values = group[value_attribute].values

        imputed_values = compute_outlier_imputation(values, upper_percentile, lower_percentile)

        data.loc[index, value_attribute] = imputed_values

    data = data.dropna(subset=[value_attribute])

    return data

In [24]:
within_72 = outlier_imputation(within_72, 'itemid', 'value', num_col, upper_percentile=98, lower_percentile=2)

In [ ]:
final_chart=pd.DataFrame()
t=0

for i in tqdm(np.arange(0, 72, 0.25)):  
    sub_chart = within_72[(within_72['start_time'] >= i) & (within_72['start_time'] < i + 0.25)]
    
    sub_chart_num = sub_chart[sub_chart['itemid'].isin(num_col)].groupby(['stay_id', 'itemid']).agg({'value': np.nanmean}).reset_index()

    sub_chart_str = sub_chart[sub_chart['itemid'].isin(str_col)].groupby(['stay_id', 'itemid']).agg({'value': lambda x: list(x)}).reset_index()

    sub_chart = pd.concat([sub_chart_num, sub_chart_str], ignore_index=True)

    sub_chart['start_time'] = t

    if final_chart.empty:
        final_chart = sub_chart
    else:
        final_chart = final_chart.append(sub_chart)

    t += 1

100%|████████████████████████████████████████████████████████████████████████████████| 288/288 [04:18<00:00,  1.11it/s]


In [28]:
final_chart.shape

(1426525, 4)

In [30]:
final_chart.head(2)

,stay_id,itemid,value,start_time
0,30000831,50802,2.0,0
1,30000831,51249,32.6,0


In [31]:
for it in final_chart.itemid.unique():
    print(it,d_labitems[d_labitems.itemid==it].label.values)
    print(final_chart[final_chart.itemid==it].value.value_counts())

50802 ['Base Excess']
 0.000000     53581
-1.000000     25867
-2.000000     23111
-3.000000     21322
 1.000000     18957
              ...  
-3.600000         1
 14.500000        1
-2.333333         1
 7.666667         1
 0.600000         1
Name: value, Length: 205, dtype: int64
51249 ['MCHC']
33.20    8309
33.30    8080
33.10    7307
32.90    7300
32.80    7232
         ... 
24.00       1
26.80       1
31.80       1
29.85       1
31.25       1
Name: value, Length: 248, dtype: int64
51277 ['RDW']
13.20    8266
14.60    8076
15.90    6130
13.70    5705
14.30    5697
         ... 
32.10       1
15.15       1
17.65       1
33.80       1
17.30       1
Name: value, Length: 319, dtype: int64
51279 ['Red Blood Cells']
3.07    1738
3.02    1694
3.23    1690
3.36    1688
3.14    1688
        ... 
0.74       1
6.55       1
6.65       1
2.97       1
2.28       1
Name: value, Length: 754, dtype: int64
52172 ['RDW-SD']
46.50     1953
47.80     1924
49.10     1800
45.10     1781
50.40     1704
    

In [32]:
d_labitems[d_labitems.itemid.isin(final_chart.itemid.unique())]

,itemid,label,fluid,category
1,50802,Base Excess,Blood,Blood Gas
9,50811,Hemoglobin,Blood,Blood Gas
434,51249,MCHC,Blood,Hematology
462,51277,RDW,Blood,Hematology
464,51279,Red Blood Cells,Blood,Hematology
485,51300,WBC Count,Blood,Hematology
623,51463,Bacteria,Urine,Hematology
653,51493,RBC,Urine,Hematology
676,51516,WBC,Urine,Hematology
1261,52172,RDW-SD,Blood,Hematology


In [33]:
process_final_chart = final_chart.copy()

In [34]:
num_final_chart = process_final_chart[~process_final_chart.itemid.isin(str_col)]
num_final_chart.shape

(1423751, 4)

In [35]:
num_final_chart.head(2)

,stay_id,itemid,value,start_time
0,30000831,50802,2.0,0
1,30000831,51249,32.6,0


In [ ]:
str_final_chart = process_final_chart[process_final_chart.itemid.isin(str_col)]
str_final_chart['value'] = str_final_chart['value'].apply(lambda x: x[-1] if isinstance(x, list) else x)
str_final_chart.shape

(2774, 4)

In [37]:
str_final_chart.head(2)

,stay_id,itemid,value,start_time
16875,30085420,51463,FEW,0
16876,30334209,51463,OCC,0


In [38]:
process_final_chart = pd.concat([num_final_chart,str_final_chart])
process_final_chart.shape

(1426525, 4)

In [101]:
del(num_final_chart,str_final_chart)

In [39]:
process_final_chart.tail(2)

,stay_id,itemid,value,start_time
1657,38268306,51463,MOD,287
1658,38852310,51463,___,287


In [ ]:
los=288
def create_Dict(chart):
    dataDic = {}
    feat = chart['itemid'].unique()  
    
    for hid in tqdm(chart.stay_id.unique()):  
        df2 = chart[chart['stay_id'] == hid]  
        
        if df2.shape[0] == 0:
            print(f'空：{hid}')
            val = pd.DataFrame(np.zeros([los, len(feat)]), columns=feat) 
            val = val.fillna(0)  

        else:

            val = df2.pivot(index='start_time', columns='itemid', values='value')
            

            add_indices = pd.Index(range(los)).difference(val.index)
            add_df = pd.DataFrame(index=add_indices, columns=val.columns).fillna(0)  
            val = pd.concat([val, add_df])  
            val = val.sort_index()  
            

            feat_df = pd.DataFrame(columns=list(set(feat) - set(val.columns)))  
            val = pd.concat([val, feat_df], axis=1) 
            
            val = val[feat]  
            val = val.fillna(0)  

        val.to_csv(f'/MIMIC_IV/Final_Lab/{hid}.csv', index=False)

In [41]:
create_Dict(process_final_chart)

100%|████████████████████████████████████████████████████████████████████████████| 74331/74331 [23:39<00:00, 52.38it/s]


In [ ]:
def add_dynamic_process(dynamic, str_col):
    dynamic.replace(0, np.nan, inplace=True)
    
    dynamic.dropna(axis=1, how='all', inplace=True)
    
    dynamic.columns = [
        f"LAB_Str_{col}" if int(col) in str_col else 
        f"LAB_{col}" 
        for col in dynamic.columns
    ]
    
    for col in dynamic.columns:
        dynamic[col] = dynamic[col].ffill().bfill()
    
    return dynamic

In [ ]:

save_path = '/MIMIC_IV/Final_Lab_2/'

for i in tqdm(os.listdir('/MIMIC_IV/Final_Lab/')):
    try:
        file_path = f'/MIMIC_IV/Final_Lab/{i}'
        dynamic = pd.read_csv(file_path)
        dynamic = add_dynamic_process(dynamic, str_col)

        save_file_path = os.path.join(save_path, i)
        dynamic.to_csv(save_file_path, index=False)
    
    except Exception as e:

        print(f"Error processing file {i}: {e}")

100%|████████████████████████████████████████████████████████████████████████████| 74331/74331 [28:28<00:00, 43.50it/s]


In [ ]:
cols_count = Counter()

def median_ts_process(ts):
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  
    
    processed_values = {}
    for col in ts.columns:
        if col.startswith('LAB_Cate_'):  
            processed_values[col] = ts[col].max()
        elif col.startswith('LAB_Str_'):  
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            processed_values[col] = ts[col].median()
    
    result = pd.DataFrame([processed_values])
    
    return result


In [ ]:
all_ts = pd.DataFrame()

for i in tqdm(os.listdir('/MIMIC_IV/Final_Lab_2/')):
    try:

        file_path = f'/MIMIC_IV/Final_Lab_2/{i}'
        ts = pd.read_csv(file_path)


        ts = median_ts_process(ts)

        cols_count.update(ts.columns)

        ts['ICUSTAY_ID'] = i.split('.')[0]


        all_ts = pd.concat([all_ts, ts], ignore_index=True)
        
    except Exception as e:

        print(f"Error processing file {i}: {e}")
    
all_ts = all_ts.dropna(axis=1, how='all')

  9%|██████▉                                                                      | 6731/74331 [02:34<21:36, 52.14it/s]

Error processing file 30894721.csv: No columns to parse from file


 24%|██████████████████▍                                                         | 18014/74331 [05:58<17:14, 54.42it/s]

Error processing file 32419796.csv: No columns to parse from file


 33%|█████████████████████████▎                                                  | 24701/74331 [08:06<15:46, 52.44it/s]

...

 68%|███████████████████████████████████████████████████▎                        | 50217/74331 [18:40<12:17, 32.69it/s]

Error processing file 36726295.csv: No columns to parse from file


 71%|█████████████████████████████████████████████████████▉                      | 52790/74331 [19:39<06:54, 52.01it/s]

Error processing file 37077064.csv: No columns to parse from file


100%|████████████████████████████████████████████████████████████████████████████| 74331/74331 [27:38<00:00, 44.83it/s]


In [46]:
all_ts['LAB_Str_51463'] = all_ts['LAB_Str_51463'].replace(['___', 'NONE'], 'None')

In [49]:
all_ts = all_ts[['ICUSTAY_ID','LAB_50802', 'LAB_51249', 'LAB_51277', 'LAB_51279', 
       'LAB_52172', 'LAB_51516', 'LAB_50811', 'LAB_51493', 'LAB_51300','LAB_Str_51463']]

In [50]:
all_ts.shape

(74321, 11)

In [ ]:
all_ts.to_csv('/MIMIC_IV/All_Lab_mean_max_mode_nan.csv',index=False)